# 11. Seleccion de leads: global vs estratificada por regional — Fase 6

La regional se **conserva** en el modelo (`modelo_fase6_XGBoost-sin-avicena`, 58 feats): quitarla
costaba ~20% de AUC-PR y no arreglaba la equidad (la geografia se colaba via features proxy,
notebook 10). La equidad geografica se atiende en **como se asignan los leads**:

- **Global**: top 10% de riesgo absoluto en toda la EPS. Maximiza recall total pero deja que
  las regionales con mas datos (Bogota) acaparen cupos.
- **Estratificada**: top 10% **dentro de cada regional**. Cada regional recibe leads proporcionales
  a su poblacion -> equidad por diseno. Cuesta algo de recall total (no concentra en los de mayor
  riesgo absoluto), pero es defendible y operacionalmente justo para el A/B.

Se cuantifica el trade-off recall vs equidad.


In [1]:
import json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import xgboost as xgb, joblib
from sklearn.metrics import average_precision_score

B = "bases"; RNG = 42
model = joblib.load(f"{B}/modelo_fase6_XGBoost-sin-avicena.joblib")
nat_va = pd.read_parquet(f"{B}/prediccion_mama_val_native.parquet")
DROP = ['tiene_avicena']
FEAT = [c for c in nat_va.columns if c not in ('key', 'label') and c not in DROP]
assert model.n_features_in_ == len(FEAT), (model.n_features_in_, len(FEAT))
y = nat_va['label'].values.astype(int)
X = nat_va[FEAT].astype('float32').values
p = model.predict_proba(X)[:, 1]
print(f"val {X.shape} pos {y.sum()} | AUC-PR {average_precision_score(y, p):.4f}")

# regional por fila desde dummies REG_*
REG = [c for c in nat_va.columns if c.startswith('REG_')]
reg_label = nat_va[REG].values.argmax(axis=1)
reg_names = [c.replace('REG_', '').strip() for c in REG]
regional = np.array([reg_names[i] for i in reg_label])
print("regionales:", reg_names)


val (2410807, 58) pos 993 | AUC-PR 0.0388


regionales: ['1 REG BOGOTA', '2 REG CALI', '3 REG BARRANQUILLA', '4 REG BUCARAMANGA', '5 REG MEDELLIN', '6 REG CENTRO ORIENTE']


## 1. Seleccion global vs estratificada (top 10%)

In [2]:
FRAC = 0.10

# GLOBAL: top 10% absoluto
k_global = max(1, int(len(p) * FRAC))
sel_global = np.zeros(len(p), dtype=bool)
sel_global[np.argpartition(-p, k_global)[:k_global]] = True

# ESTRATIFICADA: top 10% dentro de cada regional
sel_strat = np.zeros(len(p), dtype=bool)
for r in reg_names:
    mask = regional == r
    idx = np.where(mask)[0]
    kk = max(1, int(len(idx) * FRAC))
    top_local = idx[np.argpartition(-p[idx], kk)[:kk]]
    sel_strat[top_local] = True

def recall(sel):
    return y[sel].sum() / y.sum()

print(f"GLOBAL      : {sel_global.sum():>7} leads | recall {recall(sel_global):.4f}")
print(f"ESTRATIFICADA: {sel_strat.sum():>7} leads | recall {recall(sel_strat):.4f}")
print(f"\nTrade-off recall: {recall(sel_strat)-recall(sel_global):+.4f} "
      f"({(recall(sel_strat)/recall(sel_global)-1)*100:+.1f}%)")


GLOBAL      :  241080 leads | recall 0.6989
ESTRATIFICADA:  241078 leads | recall 0.6928

Trade-off recall: -0.0060 (-0.9%)


## 2. Composicion regional de los leads (equidad)

`sobre_representacion` = % de leads de la regional / % de poblacion de la regional. ~1 = justo.


In [3]:
def composicion(sel, nombre):
    pob = pd.Series(regional).value_counts(normalize=True)
    leads = pd.Series(regional[sel]).value_counts(normalize=True)
    tab = pd.DataFrame({'%_leads': leads * 100, '%_poblacion': pob.reindex(leads.index) * 100})
    tab['sobre_repr'] = tab['%_leads'] / tab['%_poblacion']
    tab = tab.sort_values('%_poblacion', ascending=False)
    print(f"=== {nombre} ===")
    print(tab.to_string(float_format='{:.2f}'.format))
    print(f"  dispersion sobre_repr (max-min): {tab['sobre_repr'].max()-tab['sobre_repr'].min():.2f}\n")
    return tab

pd.set_option('display.width', 200)
tab_g = composicion(sel_global, 'GLOBAL (top 10% absoluto)')
tab_s = composicion(sel_strat, 'ESTRATIFICADA (top 10% por regional)')
print("La estratificada lleva sobre_repr ~1 en todas las regionales (cuota proporcional);")
print("la global concentra cupos donde hay mas datos.")


=== GLOBAL (top 10% absoluto) ===
                      %_leads  %_poblacion  sobre_repr
1 REG BOGOTA            30.91        36.61        0.84
6 REG CENTRO ORIENTE    11.64        18.38        0.63
2 REG CALI              15.93        13.57        1.17
4 REG BUCARAMANGA       11.58        11.96        0.97
3 REG BARRANQUILLA      14.53        11.73        1.24
5 REG MEDELLIN          15.42         7.74        1.99
  dispersion sobre_repr (max-min): 1.36



=== ESTRATIFICADA (top 10% por regional) ===
                      %_leads  %_poblacion  sobre_repr
1 REG BOGOTA            36.61        36.61        1.00
6 REG CENTRO ORIENTE    18.38        18.38        1.00
2 REG CALI              13.57        13.57        1.00
4 REG BUCARAMANGA       11.96        11.96        1.00
3 REG BARRANQUILLA      11.73        11.73        1.00
5 REG MEDELLIN           7.74         7.74        1.00
  dispersion sobre_repr (max-min): 0.00

La estratificada lleva sobre_repr ~1 en todas las regionales (cuota proporcional);
la global concentra cupos donde hay mas datos.


## 3. Recall por regional (equidad de cobertura)

No solo cupos justos: cuanto de los casos reales de CADA regional se captura.


In [4]:
rows = []
for r in reg_names:
    mask = regional == r
    pos_r = y[mask].sum()
    if pos_r == 0:
        continue
    rg = y[mask & sel_global].sum() / pos_r
    rs = y[mask & sel_strat].sum() / pos_r
    rows.append({'regional': r, 'positivos': int(pos_r),
                 'recall_global': rg, 'recall_estratif': rs})
rec = pd.DataFrame(rows).set_index('regional').sort_values('positivos', ascending=False)
print("=== Recall de casos reales por regional ===")
print(rec.to_string(float_format='{:.4f}'.format))
print(f"\nDispersion recall GLOBAL  (max-min): {rec['recall_global'].max()-rec['recall_global'].min():.4f}")
print(f"Dispersion recall ESTRATIF (max-min): {rec['recall_estratif'].max()-rec['recall_estratif'].min():.4f}")
print("Menor dispersion = cobertura mas pareja entre regionales.")


=== Recall de casos reales por regional ===
                      positivos  recall_global  recall_estratif
regional                                                       
1 REG BOGOTA                376         0.9096           0.9202
6 REG CENTRO ORIENTE        150         0.3800           0.4733
2 REG CALI                  139         0.6619           0.6403
3 REG BARRANQUILLA          122         0.5902           0.5738
4 REG BUCARAMANGA           114         0.7018           0.7018
5 REG MEDELLIN               92         0.5543           0.3478

Dispersion recall GLOBAL  (max-min): 0.5296
Dispersion recall ESTRATIF (max-min): 0.5724
Menor dispersion = cobertura mas pareja entre regionales.


## 4. Guardar diagnostico

In [5]:
out = {
    'modelo': 'XGBoost-sin-avicena (regional conservada)',
    'frac': FRAC,
    'recall_global': float(recall(sel_global)),
    'recall_estratificada': float(recall(sel_strat)),
    'trade_off_recall': float(recall(sel_strat) - recall(sel_global)),
    'composicion_global': tab_g.round(3).reset_index().to_dict(orient='records'),
    'composicion_estratificada': tab_s.round(3).reset_index().to_dict(orient='records'),
    'recall_por_regional': rec.round(4).reset_index().to_dict(orient='records'),
}
with open(f"{B}/seleccion_estratificada.json", "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False)
print("Guardado: bases/seleccion_estratificada.json")
print(f"\nGlobal recall {recall(sel_global):.4f} vs estratificada {recall(sel_strat):.4f}")
print("Recomendacion: estratificada para el A/B (equidad por diseno, recall casi igual).")


Guardado: bases/seleccion_estratificada.json

Global recall 0.6989 vs estratificada 0.6928
Recomendacion: estratificada para el A/B (equidad por diseno, recall casi igual).
